# 实验一：静态页面与 API 数据采集

本 Notebook 采集全部 Device API 数据，并通过 HTML 分页采集 Company 前 3 页；完成字段解析、数量与 ID 校验、Company–Device 一致性检查，并将结果保存到 `data/`。

In [1]:
from collections import Counter
from concurrent.futures import ThreadPoolExecutor
import json
from pathlib import Path
import re
from urllib.parse import urljoin

import requests
from bs4 import BeautifulSoup

BASE_URL = "https://115.159.159.82"
API_URL = urljoin(BASE_URL, "/api/devices")
TIMEOUT = 20
session = requests.Session()

def get_json_page(page, page_size):
    response = session.get(
        API_URL,
        params={"page": page, "page_size": page_size},
        timeout=TIMEOUT,
    )
    response.raise_for_status()
    result = response.json()
    if not isinstance(result, dict) or "items" not in result:
        raise ValueError(f"Unexpected API response on page {page}: {result!r}")
    return result

def get_html_page(url):
    response = session.get(url, timeout=TIMEOUT)
    response.raise_for_status()
    return response.text


## 1. Device：检查 API 并采集所有分页

设备列表由 `/api/devices` 返回 JSON。先读取第一页的分页字段，再用 `total` 和 `page_size` 计算总页数。

In [2]:
first_page = get_json_page(page=1, page_size=50)
total = first_page["total"]
page_size = first_page["page_size"]
total_pages = (total + page_size - 1) // page_size

if first_page.get("page") != 1:
    raise ValueError(f"Expected page 1, got {first_page.get('page')!r}")

print("Request URL:", API_URL)
print("Request Method: GET")
print("Pagination fields: page, page_size, total, items")
print("Page size:", page_size)
print("Device total:", total)
print("Calculated total pages:", total_pages)
print("First device:", first_page["items"][0])


Request URL: https://115.159.159.82/api/devices
Request Method: GET
Pagination fields: page, page_size, total, items
Page size: 50
Device total: 5000
Calculated total pages: 100
First device: {'device_id': 'D00000001', 'company_id': 'C000001', 'device_name': '循环泵-000001-001', 'device_type': 'pump', 'model': 'SP-PUM-800', 'zone': '智造一区', 'install_date': '2021-09-19', 'rated_power_kw': 71.2, 'sampling_interval_s': 30, 'protocol': 'mqtt', 'status': 'running', 'updated_at': '2026-09-01T03:58:00'}


In [3]:
def fetch_device_page(page):
    page_data = get_json_page(page=page, page_size=page_size)
    if page_data.get("page") != page:
        raise ValueError(f"Requested page {page}, received {page_data.get('page')!r}")
    if page_data.get("page_size") != page_size or page_data.get("total") != total:
        raise ValueError(f"Pagination metadata changed on page {page}")
    return page_data["items"]

devices = list(first_page["items"])
with ThreadPoolExecutor(max_workers=8) as executor:
    for page_items in executor.map(fetch_device_page, range(2, total_pages + 1)):
        devices.extend(page_items)

device_ids = [item["device_id"] for item in devices]
if len(devices) != total:
    raise ValueError(f"Expected {total} devices, collected {len(devices)}")
if len(set(device_ids)) != len(device_ids):
    raise ValueError("Duplicate device_id values were found")

print("Collected devices:", len(devices))
print("Unique device_id:", len(set(device_ids)))


Collected devices: 5000
Unique device_id: 5000


## 2. Company：解析静态 HTML 并跟随“下一页”

用 `article.company-card` 选取企业记录；从 `data-company-id`、标题、状态和 `dt/dd` 字段提取信息。通过分页区的链接继续采集，直到采满实验要求的 3 页。

In [4]:
def parse_company_card(card):
    name_node = card.select_one("h2 a")
    status_node = card.select_one(".badge")
    detail_nodes = card.select("dl dd")
    if name_node is None or status_node is None or len(detail_nodes) < 4:
        raise ValueError(f"Unexpected company card structure: {card}")

    device_count_text = detail_nodes[3].get_text(strip=True)
    count_match = re.search(r"\d+", device_count_text)
    if count_match is None:
        raise ValueError(f"Cannot parse device count: {device_count_text!r}")

    return {
        "company_id": card["data-company-id"],
        "name": name_node.get_text(strip=True),
        "industry": detail_nodes[0].get_text(strip=True),
        "zone": detail_nodes[1].get_text(strip=True),
        "device_count": int(count_match.group()),
        "status": status_node.get_text(strip=True),
    }

companies = []
current_url = urljoin(BASE_URL, "/companies")
company_page_counts = []
visited_urls = set()

for page_number in range(1, 4):
    if current_url in visited_urls:
        raise ValueError(f"Pagination loop detected at {current_url}")
    visited_urls.add(current_url)

    soup = BeautifulSoup(get_html_page(current_url), "html.parser")
    cards = soup.select("article.company-card")
    if not cards:
        raise ValueError(f"No company cards found on page {page_number}")
    company_page_counts.append(len(cards))
    companies.extend(parse_company_card(card) for card in cards)

    if page_number < 3:
        next_link = soup.select_one("nav.pagination a:last-child")
        if next_link is None or not next_link.get("href"):
            raise ValueError(f"Next-page link not found after page {page_number}")
        current_url = urljoin(current_url, next_link["href"])

company_ids = [company["company_id"] for company in companies]
if len(set(company_ids)) != len(company_ids):
    raise ValueError("Duplicate company_id values were found")

print("Company records per page:", company_page_counts)
print("Companies collected:", len(companies))
print("Unique company_id:", len(set(company_ids)))
print("First company:", companies[0])


Company records per page: [20, 20, 20]
Companies collected: 60
Unique company_id: 60
First company: {'company_id': 'C000001', 'name': '临港精工001有限公司', 'industry': '汽车零部件', 'zone': '智造一区', 'device_count': 6, 'status': '运营中'}


## 3. 校验并保存结果

对已采集的 60 家企业，将网页中的设备数与完整 Device 数据中按 `company_id` 汇总的设备数比较；然后保存 JSON 文件。

In [5]:
device_counts = Counter(item["company_id"] for item in devices)
mismatches = [
    {
        "company_id": company["company_id"],
        "expected": company["device_count"],
        "actual": device_counts[company["company_id"]],
    }
    for company in companies
    if company["device_count"] != device_counts[company["company_id"]]
]

data_dir = Path("data")
data_dir.mkdir(parents=True, exist_ok=True)
with (data_dir / "companies.json").open("w", encoding="utf-8") as file:
    json.dump(companies, file, ensure_ascii=False, indent=2)
with (data_dir / "devices.json").open("w", encoding="utf-8") as file:
    json.dump(devices, file, ensure_ascii=False, indent=2)

print("Company 总数:", len(companies))
print("唯一 company_id:", len(set(company_ids)))
print("Device 总数:", len(devices))
print("唯一 device_id:", len(set(device_ids)))
print("Company–Device 不一致企业数:", len(mismatches))
print("Mismatch details:", mismatches[:10])
print("Saved:", data_dir / "companies.json", "and", data_dir / "devices.json")


Company 总数: 60
唯一 company_id: 60
Device 总数: 5000
唯一 device_id: 5000
Company–Device 不一致企业数: 0
Mismatch details: []
Saved: data\companies.json and data\devices.json
